In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Panimula sa mga primitive

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Available na ang beta release ng bagong execution model. Ang directed execution model ay nagbibigay ng mas maraming flexibility sa pag-customize ng iyong error mitigation workflow. Tingnan ang gabay na [Directed execution model](/guides/directed-execution-model) para sa karagdagang impormasyon.

<span id="qpu-access-patterns"></span>
## Bakit nagpakilala ng mga primitive ang Qiskit?
Katulad ng mga unang araw ng classical computers, nang kailangan pang manipulahin ng mga developer ang CPU registers nang direkta, ang unang interface sa mga QPU ay nagbabalik lang ng raw data mula sa control electronics.
Hindi ito malaking problema noong nasa mga laboratoryo pa lang ang mga QPU at direktang access lang ang ibinibigay sa mga mananaliksik.
Kinikilala na karamihan sa mga developer ay hindi kailangang maging pamilyar sa pag-distill ng ganoong raw data sa 0s at 1s, ipinakilala ng Qiskit ang `backend.run`, isang unang abstraction para sa pag-access sa mga QPU sa cloud. Nagbigay-daan ito sa mga developer na gumana sa isang pamilyar na format ng data at mas mapagtuunan ang mas malaking larawan.

Habang lumalawak ang access sa mga QPU, at habang mas maraming quantum algorithm ang binubuo,
lumitaw muli ang pangangailangan para sa mas mataas na antas ng abstraction. Bilang tugon, ipinakilala ng Qiskit ang
primitives interface, na na-optimize para sa dalawang pangunahing gawain sa pagbuo ng quantum algorithm:
pagtatantya ng expectation value (`Estimator`) at pag-sample ng circuit (`Sampler`). Ang layunin ay
muli na tulungan ang mga developer na mas mapagtuunan ang inobasyon at hindi ang conversion ng data. Pinapalitan ng primitives interface ang `backend.run` interface, dahil nagbibigay ang `Sampler` ng parehong direktang hardware access na inaalok noon ng `backend.run`.

## Ano ang isang primitive?
Ang mga computing system ay binubuo ng maraming layer ng abstraction. Ang mga abstraction ay nagbibigay-daan sa iyo na magfocus sa isang partikular na antas ng detalye na may kaugnayan sa gawain. Habang lalapit ka sa hardware, mas mababang antas ng abstraction ang kailangan mo (halimbawa, maaaring kailangang ilipat o manipulahin ang data sa antas ng CPU instruction). Habang mas kumplikado ang gawain na nais mong gawin, mas mataas ang antas ng mga abstraction (halimbawa, maaaring gumagamit ka ng programming library para magsagawa ng algebraic calculations).

Sa kontekstong ito, ang isang *primitive* ay ang pinakamaliit na processing instruction, ang pinakasimpleng building block mula sa kung saan maaaring lumikha ng isang bagay na kapaki-pakinabang para sa isang partikular na antas ng abstraction.

Ang kamakailang pag-unlad sa quantum computing ay nagpataas ng pangangailangan na magtrabaho sa mas mataas na antas ng abstraction.
Habang ang larangan ay gumagalaw patungo sa mas malalaking quantum processing units (QPU) at mas kumplikadong mga workflow, lumilipat ang pokus mula sa pakikipag-ugnayan sa mga indibidwal na qubit signal patungo sa pagtingin sa mga quantum device bilang mga sistema na nagsasagawa ng mga kinakailangang gawain.

Ang dalawang pinakakaraniwang gawain para sa mga quantum computer ay ang pag-sample ng mga quantum state at pagkalkula ng mga expectation value.
Ang mga gawaing ito ang nagbigay-inspirasyon sa disenyo ng mga Qiskit primitive: **Estimator** at **Sampler**.

- Kinakalkula ng Estimator ang mga expectation value ng mga observable kaugnay ng mga state na inihanda ng mga quantum circuit.
- Sino-sample ng Sampler ang output register mula sa pagpapatupad ng quantum circuit.

Sa madaling salita, ang computational model na ipinakilala ng mga Qiskit primitive ay nagdadala sa quantum programming ng isang hakbang na mas malapit sa kinaroroonan ng classical programming ngayon, kung saan ang pokus ay mas kaunti sa mga detalye ng hardware at mas marami sa mga resulta na nais mong makamit.

## Kahulugan at mga implementasyon ng primitive
Mayroong dalawang uri ng Qiskit primitive: ang mga base class, at ang kanilang mga implementasyon. Ang mga Qiskit primitive ay tinukoy ng mga open-source primitive base class na nasa Qiskit SDK (sa [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) module). Ang mga provider (tulad ng Qiskit Runtime) ay maaaring gumamit ng mga base class na ito para makuha ang kanilang sariling mga implementasyon ng Sampler at Estimator. Karamihan sa mga user ay makikipag-ugnayan sa mga implementasyon ng provider, hindi sa mga base primitive.

### Mga base class
[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) at [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Mga abstract base class na nagtatakda ng karaniwang interface para sa pagpapatupad ng mga primitive. Lahat ng iba pang klase sa [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) module ay nagmumula sa mga base class na ito. Dapat gamitin ng mga developer ang mga ito kung interesado silang lumikha ng sarili nilang primitives-based execution model para sa isang partikular na provider. Ang mga klase na ito ay maaari ring maging kapaki-pakinabang para sa mga gustong gumawa ng lubos na customized na pagpoproseso at natuklasan na ang mga kasalukuyang implementasyon ng primitive ay masimple para sa kanilang mga pangangailangan. Ang mga pangkalahatang user ay hindi direktang gagamit ng mga base class.

<span id="implementations"></span>
### Mga implementasyon
Ito ang mga implementasyon ng mga primitives base class:

- Ang mga Qiskit Runtime primitive ([`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) at [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)) ay nagbibigay ng mas sopistikadong implementasyon (halimbawa, sa pamamagitan ng pagsasama ng error mitigation) bilang isang cloud-based na serbisyo. Ang implementasyong ito ng mga base primitive ay ginagamit para ma-access ang IBM Quantum&reg; hardware. Ina-access ang mga ito sa pamamagitan ng IBM Qiskit Runtime.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) at [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Mga reference implementation ng mga primitive na gumagamit ng simulator na built-in sa Qiskit. Ginawa ang mga ito gamit ang Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) module, na gumagawa ng mga resulta batay sa ideal statevector simulations. Ina-access ang mga ito sa pamamagitan ng Qiskit.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) at [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Magagamit mo ang mga klase na ito para "i-wrap" ang anumang quantum computing resource sa isang primitive. Nagbibigay-daan ito sa iyo na sumulat ng primitive-style na code para sa mga provider na wala pang primitives-based na interface. Ang mga klase na ito ay magagamit tulad ng regular na Sampler at Estimator, maliban na dapat silang simulan na may karagdagang `backend` na argument para sa pagpili kung aling quantum computer ang pagtatakbuhin. Ina-access ang mga ito sa pamamagitan ng paggamit ng Qiskit.

## Mga benepisyo ng mga Qiskit primitive
Sa mga primitive, ang mga Qiskit user ay maaaring sumulat ng quantum code para sa isang partikular na QPU nang hindi kailangang tahasang pamahalaan ang bawat detalye. Gayundin, dahil sa karagdagang layer ng abstraction, maaari kang mas madaling ma-access ang mga advanced na kakayahan ng hardware ng isang partikular na provider. Halimbawa, sa mga Qiskit Runtime primitive, maaari kang makinabang mula sa mga pinakabagong pagsulong sa error mitigation at suppression sa pamamagitan ng pag-toggle ng mga opsyon tulad ng [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) ng primitive, sa halip na bumuo ng sariling implementasyon ng mga pamamaraang ito.

Para sa mga hardware provider, ang katutubong pagpapatupad ng mga primitive ay nangangahulugang maaari mong bigyan ang iyong mga user ng mas "out-of-the-box" na paraan para ma-access ang iyong mga hardware feature tulad ng mga advanced na post-processing technique. Kaya naman, mas madali para sa iyong mga user na makinabang mula sa pinakamahusay na kakayahan ng iyong hardware.

## Mga detalye ng primitive
Tulad ng inilarawan kanina, lahat ng primitive ay nilikha mula sa mga base class; samakatuwid, mayroon silang parehong pangkalahatang istraktura at paggamit. Halimbawa, ang format ng input para sa lahat ng Estimator primitive ay pareho. Gayunpaman, may mga pagkakaiba sa mga implementasyon na nagpapakaiba sa kanila.

> **Note:** Dahil karamihan sa mga user ay nag-a-access ng mga Qiskit Runtime primitive, ang mga halimbawa sa natitirang bahagi ng seksyong ito ay batay sa mga Qiskit Runtime primitive.

<span id="estimator"></span>
### Estimator
Kinakalkula ng Estimator primitive ang mga expectation value para sa isa o higit pang observable kaugnay ng mga state na inihanda ng mga quantum circuit. Ang mga circuit ay maaaring parametrized, hangga't ang mga parameter value ay ibinibigay din bilang input sa primitive.

Ang input ay isang array ng [mga PUB.](/guides/primitive-input-output#pubs) Ang bawat PUB ay nasa format na:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

kung saan ang optional na `parameter values` ay maaaring isang listahan o isang parameter. Sinusuportahan ng iba't ibang implementasyon ng Estimator ang iba't ibang configuration option. Kung ang input ay naglalaman ng mga measurement, ito ay binabalewala.

Ang output ay isang [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) na naglalaman ng mga kinakalkula na expectation value bawat pares, at ang kanilang mga standard error, sa `PubResult` na anyo. Ang bawat `PubResult` ay naglalaman ng parehong data at metadata.

Pinagsasama ng Estimator ang mga elemento mula sa mga observable at parameter value sa pamamagitan ng pagsunod sa mga NumPy broadcasting rule tulad ng inilarawan sa paksa na [Primitive inputs and outputs](primitive-input-output#broadcasting-rules).

Halimbawa:

In [1]:
# This cell is hidden from users, it creates the circuits and observables to run

from qiskit_ibm_runtime import EstimatorV2, SamplerV2, QiskitRuntimeService
from qiskit.circuit.random import random_circuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
import numpy as np

service = QiskitRuntimeService()
backend = service.least_busy()
phi = Parameter("phi")

circuit1 = random_circuit(10, 5, seed=12345)
circuit1.rzz(phi, 1, 2)
observable1 = SparsePauliOp.from_sparse_list(
    [("ZXYZ", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values1 = np.random.uniform(size=5).T

circuit2 = random_circuit(10, 5, seed=12345)
circuit2.rzz(phi, 1, 2)
observable2 = SparsePauliOp.from_sparse_list(
    [("XZYX", [1, 2, 3, 4], 1)], num_qubits=10
)
param_values2 = np.random.uniform(size=5).T

shots1 = 164
shots2 = 1024

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
circuit1 = pm.run(circuit1)
circuit2 = pm.run(circuit2)
observable1 = observable1.apply_layout(circuit1.layout)
observable2 = observable2.apply_layout(circuit2.layout)

In [2]:
estimator = EstimatorV2(mode=backend)
estimator_job = estimator.run(
    [
        (circuit1, observable1, param_values1),
        (circuit2, observable2, param_values2),
    ]
)

<span id="sampler"></span>
### Sampler
Ang pangunahing gawain ng Sampler ay ang pag-sample ng output register mula sa pagpapatupad ng isa o higit pang quantum circuit. Ang mga input circuit ay maaaring parametrized, hangga't ang mga parameter value ay ibinibigay din bilang input sa primitive.

Ang input ay isa o higit pang [PUB,](/guides/primitive-input-output#pubs) sa format na:

(`<single circuit>`, `<one or more optional parameter value>`, `<optional shots>`),

kung saan maaaring maraming `parameter values` na item, at ang bawat item ay maaaring isang array o isang parameter, depende sa napiling circuit. Bukod dito, ang input ay dapat naglalaman ng mga measurement.

Ang output ay mga bilang o per-shot na measurement, bilang mga [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult) na object, nang walang timbang. Ang result class, gayunpaman, ay may mga pamamaraan para magbalik ng mga weighted sample, tulad ng mga bilang. Tingnan ang [Primitive inputs and outputs](primitive-input-output#broadcasting-rules) para sa kumpletong detalye.

Halimbawa:

In [3]:
# This cell is hidden from users, add measurement instructions to circuits
circuit1.measure_active()
circuit2.measure_active()

In [4]:
sampler = SamplerV2(mode=backend)
sampler_job = sampler.run(
    [
        (circuit1, param_values1, shots1),
        (circuit2, param_values2, shots2),
    ]
)

## Mga susunod na hakbang
> **Tip:** - Basahin ang [Get started with primitives](get-started-with-primitives) para maipatupad ang mga primitive sa iyong trabaho.
>     - Suriin ang mga detalyadong [halimbawa ng primitive.](primitives-examples)
>     - Mag-practice sa mga primitive sa pamamagitan ng pagtratrabaho sa [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) sa IBM Quantum Learning.
>     - Tingnan ang [EstimatorV2 API reference](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) at [SamplerV2 API reference](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2).
>     - Basahin ang [Migrate to V2 primitives](/guides/v2-primitives).